# 📓 Notebook 2｜形狀特徵：Hu/Zernike 矩・傅立葉描述子・鏈碼・幾何特徵

> 對應講義 **Part 3–4**（知識地圖站 3–4、7 選修）
>
> 主題：怎麼把「物體的輪廓形狀」變成不怕旋轉/縮放/移動的數字。

## Step 1｜合成幾種二值形狀（二值 = 1 前景 / 0 背景）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.draw import polygon
from skimage.measure import label, regionprops

>>> 把「產生形狀」和「畫圖」拆成不同格子：之後改形狀參數，不用重跑 import。
>>> 標準參數式：星星 = 週期極座標多邊形；愛心 = 心形曲線 x=16sin³t, y=13cost−5cos2t−2cos3t−cos4t。

In [ ]:
N = 128
def shape_matrix(pts_c, pts_r):
    m = np.zeros((N, N), dtype=int)    # 二值形狀用 int！label() 與 bitwise_or 才不會出錯
    rr, cc = polygon(pts_r, pts_c)
    rr = np.clip(rr, 0, N - 1); cc = np.clip(cc, 0, N - 1)
    m[rr, cc] = 1
    return m

# 星星：5 角星 = 10 個頂點，外角半徑 R 與內角半徑 r 交替（先外後內）
angs = np.pi / 2 + np.arange(10) * np.pi / 5
rads = np.array([30, 13] * 5)
star = shape_matrix(64 + rads * np.cos(angs), 64 - rads * np.sin(angs))

# 愛心：經典心形參數曲線
th = np.linspace(0, 2 * np.pi, 64)
hx = 16 * np.sin(th) ** 3
hy = 13 * np.cos(th) - 5 * np.cos(2 * th) - 2 * np.cos(3 * th) - np.cos(4 * th)
heart = shape_matrix(64 + hx * 1.9, 64 - hy * 1.9)

shapes = {'星星': star, '愛心': heart}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (name, m) in zip(axes, shapes.items()):
    ax.imshow(m, cmap='gray'); ax.set_title(name); ax.axis('off')
axes[2].imshow(star | heart, cmap='gray'); axes[2].set_title('疊在一起'); axes[2].axis('off')
plt.show()

## Step 2｜幾何特徵一把抓：regionprops（課本 7.3.4）

In [ ]:
regions = regionprops(label(star))[0]
print('面積 area          =', regions.area)
print('周長 perimeter     =', regions.perimeter)
print('圓度 γ = P²/4πA    =', regions.perimeter ** 2 / (4 * np.pi * regions.area))
print('離心率 eccentricity =', regions.eccentricity)
print('方向 orientation   =', np.degrees(regions.orientation), '度')
print('質心 centroid      =', regions.centroid)
print('solidity 凸度      =', regions.solidity)   # 形狀面積 / 凸包面積
print('Hu 矩 (moments_hu) =', np.round(regions.moments_hu, 6))

## Step 3｜驗證 Hu 矩的「三不怕」：平移・縮放・旋轉（課本 7.2.3）

理論說不變，數位影像上只是「近似不變」——我們親眼看看差多少。

> 💡 為什麼用愛心示範？星星有 5 重旋轉對稱 → 奇數階中心矩全部為 0 → φ₂..φ₇≈0，看不出變不變。「對稱形狀是 Hu 矩的盲區」——這本身就是一個值得一提的知識點。

In [ ]:
from skimage.transform import rotate, rescale

def hu_of(m):
    return regionprops(label(m))[0].moments_hu

def rotated_view(img, deg):
    '''旋轉但不裁掉角落：先加 64px 零邊框 → 旋轉（最近鄰）→ 裁回中央'''
    p = np.pad(img, 64)
    t = rotate(p, deg, order=0, preserve_range=True)
    return t[64:192, 64:192]

# 用「愛心」示範：星星有 5 重旋轉對稱，奇數階矩天然為 0，看不出變不變
base   = hu_of(heart)
shift  = hu_of(np.roll(np.roll(heart, 15, axis=0), 10, axis=1))   # 平移
small  = hu_of(rescale(heart, 0.7, order=0, preserve_range=True)) # 縮小
turned = hu_of(rotated_view(heart, 37))                            # 旋轉 37°

print(f'{"φ":>6s}  {"原始":>10s} {"平移":>10s} {"縮放":>10s} {"旋轉37°":>10s}')
for i in range(7):
    print(f'φ{i+1:>4d}  {base[i]:10.5f} {shift[i]:10.5f} {small[i]:10.5f} {turned[i]:10.5f}')

### ✏️ 練習
1. 哪個 φ 對「旋轉」最穩？對「縮放」最不穩？（提示：高階矩對離散化誤差更敏感）
2. 把 `rotate(..., resize=True)` 改為 `resize=False`（會裁切），誤差會變大嗎？為什麼？

## Step 4｜Zernike 矩（課本 7.2.3 後半・選修）

正交基底 → 資訊不重複、抗雜訊。`skimage` 一行搞定。

In [ ]:
import cmath, math

def zernike_radial(p, q, rho):
    '''徑向多項式 R_pq(ρ)（課本 Zernike 定義處）'''
    s = 0.0
    for k in range((p - abs(q)) // 2 + 1):
        s += ((-1) ** k * math.factorial(p - k) * rho ** (p - 2 * k)
              / (math.factorial(k) * math.factorial((p + abs(q)) // 2 - k)
                 * math.factorial((p - abs(q)) // 2 - k)))
    return s

def zernike_moments(img, degree):
    '''A_pq = (p+1)/π Σ I·V*_pq：影像對齊單位圓中心，|q|≤p 且 p-|q| 偶數'''
    H, W = img.shape
    cx, cy = (H - 1) / 2, (W - 1) / 2
    R = min(cx, cy)                                # 最小半徑歸一（圓外像素略過）
    moms = []
    for p in range(degree + 1):
        for q in range(-p, p + 1, 2):
            acc = 0.0 + 0j
            for y in range(H):
                for x in range(W):
                    rho = math.hypot((x - cx) / R, (y - cy) / R)
                    if rho > 1:
                        continue
                    theta = math.atan2((y - cy) / R, (x - cx) / R)
                    V = zernike_radial(p, abs(q), rho) * cmath.exp(1j * q * theta)
                    acc += img[y, x] * V.conjugate()
            moms.append(abs((p + 1) / math.pi * acc))    # |A_pq| → 旋轉不變
    return np.array(moms)

z_s = zernike_moments(star, 6)
z_h = zernike_moments(heart, 6)
print('星星 Zernike (degree≤6) 前 15 項:', np.round(z_s[:15], 4))
print('愛心 Zernike (degree≤6) 前 15 項:', np.round(z_h[:15], 4))
print('\n兩者特徵向量差很多 → 可區分性 OK ✓')

# 驗證旋轉不變性（課本 Problem 7.8：|A_pq| 對旋轉不變）
from skimage.transform import rotate
def rotated_view(img, deg):          # 與前面相同的防裁切旋轉
    p = np.pad(img, 64)
    t = rotate(p, deg, order=0, preserve_range=True)
    return t[64:192, 64:192]
zr = zernike_moments(rotated_view(star, 30), 6)
print('星星旋轉 30° 後特徵向量最大差異:', np.abs(z_s - zr).max().round(4), '（遠小於特徵量級 → 近似不變 ✓，殘差來自數位化）')

## Step 5｜傅立葉描述子：輪廓 → DFT → 用前幾個係數重建（課本 7.3.1）

In [ ]:
from skimage.measure import find_contours
from numpy.fft import fft, ifft

contour = find_contours(star, 0.5)[0]          # 形狀邊界（像素座標）
z = contour[:, 0] + 1j * contour[:, 1]          # 複數化：u_k = x_k + j y_k
F = fft(z)                                      # 傅立葉描述子 f_l
print('邊界點數:', len(z), ' 描述子個數:', len(F))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, K in zip(axes, [5, 10, 30]):
    F2 = np.zeros_like(F)
    F2[:K] = F[:K]; F2[-K:] = F[-K:]            # 只留前後 K 個（低頻）
    z2 = ifft(F2)
    ax.plot(z2.real, z2.imag, lw=2)
    ax.plot(z.real, z.imag, 'k--', lw=0.8, alpha=0.4, label='原始')
    ax.set_title(f'只用 {K*2} 個描述子重建')
    ax.axis('equal'); ax.axis('off')
    ax.legend(fontsize=8)
plt.show()

In [ ]:
# 不變性示範：看 |f_l| 對「取樣起點」的穩定性
# 旋轉邊界點（等於 DFT 的相位移動）→ 大小不變
z_shift = np.roll(z, 17)                        # 起點移動 17 格
F_shift = fft(z_shift)
print('|f_1| 原始  =', abs(F[1]).round(3))
print('|f_1| 平移起點 =', abs(F_shift[1]).round(3), '（幾乎一樣 → 大小對起點不敏感 ✓）')

## Step 6｜鏈碼 Chain Code（課本 7.3.2・簡易版）

In [ ]:
# 沿輪廓依序走，把每一步轉成 8 方向碼
DIRS = ['→', '↘', '↓', '↙', '←', '↖', '↑', '↗']
def chain_code(contour):
    codes = []
    for (x1, y1), (x2, y2) in zip(contour[:-1], contour[1:]):
        ang = np.arctan2(x2 - x1, y2 - y1)      # 課本 (7.44) 的 θn
        codes.append(int(round(ang / (np.pi / 4))) % 8)
    return codes

codes = chain_code(contour)
print('鏈碼（前 40 個）:', ''.join(str(c) for c in codes[:40]))
print('方向出現次數比例:')
hist, _ = np.histogram(codes, bins=8, range=(0, 8))
for d, h in zip(DIRS, hist):
    print(f'  方向 {d}: {h:4d} 次 ({100 * h / len(codes):4.1f}%)')

## 附錄 A｜AR 模型：驗證課本 Example 7.4（課本 7.2.4・選修）

課本：r(0)=1, r(1)=0.5, r(2)=0.85 → 解 R a = r 得 a(1)=0.1, a(2)=0.8。

In [ ]:
R = np.array([[1.0, 0.5], [0.5, 1.0]])
r = np.array([0.5, 0.85])
a = np.linalg.solve(R, r)
print('AR(2) 參數 a =', a, '→ 課本答案 [0.1, 0.8] ✓' if np.allclose(a, [0.1, 0.8]) else '✗')

# 進階：從真實合成訊號「估計」AR 參數（最小平方法 = 課本 (7.30) 的實作版）
rng = np.random.default_rng(0)
x = np.zeros(3000)
for n in range(2, len(x)):
    x[n] = 0.1 * x[n - 1] + 0.8 * x[n - 2] + rng.standard_normal()
X = np.stack([x[1:-1], x[0:-2]]).T    # 過去樣本 (x[n-1], x[n-2])
y = x[2:]                              # 現在樣本 x[n]
a_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
print('從 3000 點訊號估計出的 AR 參數:', a_hat.round(3), '（應該接近 [0.1, 0.8]）')

## 附錄 B｜Koch 雪花（課本 7.4・選修・Python 版）

In [ ]:
def koch(p1, p2, depth):
    """回傳 p1→p2 之間 Koch 曲線的折點列表"""
    if depth == 0:
        return [p1, p2]
    a = np.array(p1, float); b = np.array(p2, float)
    u = (b - a) / 3
    p = a + u; q = a + 2 * u
    mid = (p + q) / 2
    peak = mid + np.array([-u[1], u[0]]) * np.sqrt(3) / 2   # 往外凸
    return (koch(a, p, depth - 1)[:-1] + koch(p, peak, depth - 1)[:-1]
            + koch(peak, q, depth - 1)[:-1] + koch(q, b, depth - 1))

for depth in [0, 1, 2, 3]:
    pts = koch((0, 0), (1, 0), depth)
    print(f'depth={depth}: 段數={len(pts)-1:3d}  周長={len(pts)-1 * (1/3**depth):6.3f}'
          f'（初始邊長=1，理論 (4/3)^d = {(4/3)**depth:.3f}）')
    # (4/3)^d：尺越細，量得越長！

## 🏆 本筆記本小結
| 你學會了 | 對應課本 | 業界版本 |
|---|---|---|
| regionprops 幾何特徵（P/A/γ/孔洞…） | 7.3.4 | `skimage.measure.regionprops` |
| Hu 七不變矩驗證 | 7.2.3 | `skimage.measure.moments_hu` |
| Zernike 矩 | 7.2.3 | `skimage.measure.zernike_moments` |
| 傅立葉描述子 + 重建 | 7.3.1 | `numpy.fft` + `find_contours` |
| 鏈碼 + 方向統計 | 7.3.2 | 手寫（30 行內） |
| AR 參數特徵（選修） | 7.2.4 | `numpy.linalg.lstsq` |